# Grouped cross-validation by field

> Score the spectra and frozen OSSL features on fields the models have never seen, with cross-validation grouped by field, overall, by soil type and by field type

In [ ]:
#| default_exp grouped

In [ ]:
#| export
import numpy as np, pandas as pd
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import r2_score

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from fastcore.test import *
from fastai.torch_core import set_seed
from ossl_fuku_transfer.data import *
from ossl_fuku_transfer.transfer import preprocessings, load_pretrained, new_model, plain_scores
from ossl_fuku_transfer.inside import layer_features

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


The 1,389 samples of the current selection come from 391 fields, and most fields were sampled in several years. In the Cs137 study's split, used by every notebook so far, 89% of the 398 test samples come from a field that also has samples in training. A model can then partly recognise the field rather than predict from the soil, which favours models trained on Fukushima labels and can hide what OSSL adds for new fields.

This notebook scores the models with 5-fold cross-validation in which all samples of a field are in the same fold, and compares this with ordinary shuffled 5-fold cross-validation of the same samples. Every sample gets one out-of-fold prediction, so the scores by soil type and field type use all samples, not only the 341 of the fixed test set.

## Folds

`group_folds` gives each sample a fold, with every sample of a group in the same fold. Groups are dealt to folds in random order, so folds hold about the same number of groups. With one group per sample, it gives ordinary shuffled folds:

In [ ]:
#| export
def group_folds(
    groups, # Group of each sample, such as its field
    k=5, # Number of folds
    seed=0, # Seed for the order in which groups are dealt to folds
)->np.ndarray: # Fold of each sample, from 0 to `k` - 1
    "Fold of each sample, with every sample of a group in the same fold"
    u = np.random.RandomState(seed).permutation(np.unique(groups))
    fold = dict(zip(u, np.arange(len(u)) % k))
    return np.array([fold[g] for g in groups])

In [ ]:
g = np.repeat(np.arange(20), 3)
f = group_folds(g)
test_eq(pd.Series(f).groupby(g).nunique().max(), 1)
test_eq(sorted(np.unique(f)), list(range(5)))
test_eq(list(np.bincount(f)), [12] * 5)

## Out-of-fold predictions

`oof_preds` fits an estimator on all folds but one and predicts the one left out, for each fold. Within each training set, `GridSearchCV` chooses the hyperparameters on inner folds made by `group_folds` from the same groups, so no field is in both the inner training and validation sets either:

In [ ]:
#| export
def oof_preds(
    est, # Unfitted scikit-learn estimator
    grid:dict, # Parameter grid for `GridSearchCV`
    X, # Features, one row per sample
    y, # Targets
    folds, # Fold of each sample, from `group_folds`
    groups, # Group of each sample, used for the inner folds
    k=5, # Number of inner folds
)->tuple: # (out-of-fold predictions, chosen parameters of each fold)
    "Out-of-fold predictions of `est`, with hyperparameters chosen by inner cross-validation grouped by `groups`"
    p, chosen = np.empty(len(y)), []
    for f in np.unique(folds):
        tr = folds != f
        gs = GridSearchCV(est, grid, cv=PredefinedSplit(group_folds(groups[tr], k)), scoring='neg_mean_squared_error', n_jobs=-1)
        gs.fit(X[tr], y[tr])
        p[~tr] = gs.predict(X[~tr]).ravel()
        chosen.append(gs.best_params_)
    return p, chosen

In [ ]:
rng = np.random.RandomState(0)
Xs = rng.randn(200, 10)
ys = Xs @ rng.randn(10) + 0.1 * rng.randn(200)
gs_ = np.repeat(np.arange(50), 4)
ps, ch = oof_preds(make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': [0.01, 1, 100]}, Xs, ys, group_folds(gs_), gs_)
test_eq(r2_score(ys, ps) > 0.95, True)
test_eq(len(ch), 5)

## Scores by group

`field_boots` resamples whole fields with replacement, because samples of the same field are not independent:

In [ ]:
#| export
def field_boots(
    idx, # Positions of the samples to resample
    fields, # Field of every sample
    n_boot=1000, # Number of resamples
    seed=0, # Seed for the resampling
)->list: # One array of sample positions per resample
    "Resamples of `idx` that draw whole fields with replacement"
    by = pd.Series(idx).groupby(np.asarray(fields)[idx]).apply(np.asarray).to_numpy()
    rng = np.random.RandomState(seed)
    return [np.concatenate(by[rng.randint(len(by), size=len(by))]) for _ in range(n_boot)]

In [ ]:
bs = field_boots(np.arange(8), np.repeat(['a', 'b'], 4), n_boot=20)
counts_ = np.stack([np.bincount(b, minlength=8).reshape(2, 4) for b in bs])
test_eq((counts_.min(2) == counts_.max(2)).all(), True)
test_eq(counts_.sum((1, 2)).tolist(), [8] * 20)

`sample_scores` gives the R² and RMSE of each model on a set of samples, and the R² gain of each model over a reference model, with 95% intervals from `field_boots`:

In [ ]:
#| export
def sample_scores(
    y, # Observed targets of all samples
    preds:dict, # Model name: predictions of all samples
    idx, # Positions of the samples to score
    fields, # Field of every sample
    ref:str, # Model whose R² the gains are measured from
    n_boot=1000, # Number of bootstrap resamples
    seed=0, # Seed for the bootstrap
)->pd.DataFrame: # One row per model
    "R² and RMSE of each model on the samples `idx`, and the R² gain over `ref`, with 95% intervals from `field_boots`"
    y, fields = np.asarray(y), np.asarray(fields)
    boots = field_boots(idx, fields, n_boot, seed)
    b = {m: np.array([r2_score(y[i], p[i]) for i in boots]) for m, p in preds.items()}
    r2 = {m: r2_score(y[idx], p[idx]) for m, p in preds.items()}
    def ci(a, name): return dict(zip([f'{name} low', f'{name} high'], np.percentile(a, [2.5, 97.5])))
    rows = {m: {'n': len(idx), 'fields': len(np.unique(fields[idx])), 'R²': r2[m], **ci(b[m], 'R²'), 'RMSE': np.sqrt(((p[idx] - y[idx])**2).mean()),
        'gain': r2[m] - r2[ref], **ci(b[m] - b[ref], 'gain')} for m, p in preds.items()}
    return pd.DataFrame.from_dict(rows, orient='index')

`group_scores` applies `sample_scores` to all samples and to each group. Groups with fewer than `min_n` samples are left out. Within a group, R² is relative to the spread of that group, so it is lower in groups with a narrow range of the target even at the same RMSE:

In [ ]:
#| export
def group_scores(
    y, # Observed targets
    preds:dict, # Model name: predictions
    labels, # Group label of each sample, such as its soil type; NaN for unknown
    fields, # Field of each sample, the unit that the bootstrap resamples
    ref:str, # Model whose R² the gains are measured from
    n_boot=1000, # Number of bootstrap resamples
    min_n=10, # Smallest group to report
    seed=0, # Seed for the bootstrap
)->pd.DataFrame: # One row per group and model
    "`sample_scores` over all samples and within each group of `labels`"
    labels = pd.Series(labels).to_numpy()
    counts = pd.Series(labels).value_counts()
    groups = {'all': np.arange(len(y)), **{g: np.flatnonzero(labels == g) for g in counts.index[counts >= min_n]}}
    return pd.concat({g: sample_scores(y, preds, idx, fields, ref, n_boot, seed) for g, idx in groups.items()}, names=['group', 'model'])

In [ ]:
lab = np.where(np.arange(200) < 100, 'a', 'b')
gsc = group_scores(ys, {'oof': ps, 'perfect': ys}, lab, gs_, ref='oof', n_boot=50)
test_eq(gsc.loc[('all', 'perfect'), 'R²'], 1)
test_eq(list(gsc.index.get_level_values(0).unique()), ['all', 'a', 'b'])
test_eq((gsc['gain low'] <= gsc['gain']).all() and (gsc['gain'] <= gsc['gain high']).all(), True)

## Data and features

Two targets: `log1p` of Kex in cmolc/kg, as in `03_kex`, and the Cs137 study's log10 ratio of exchangeable to total Cs-137, as in `10_cs137`. Both are scored on these scales. The features, as in `10_cs137`: the spectra after SNV and a Savitzky-Golay first derivative, and block 4 of the saved OSSL Kex model, of the OSSL Kex and CEC models together, and of a model with random weights as a control:

In [ ]:
#| fukudata
fuku = load_fuku()
ids = fuku.sample_ids
props = fuku.source.reindex(ids)
ref_dir = fuku_path()/'reference'
ratio = pd.concat([pd.read_csv(ref_dir/f'cs137_ratio_{s}.csv', usecols=['ID', 'exCs137_totalCs137']) for s in ('train', 'test')])
ratio = ratio.assign(ID=ratio.ID.astype(str)).set_index('ID')['exCs137_totalCs137']
targets = {'Kex': np.log1p(props['soil_ex_K2O'].to_numpy() / mg_k2o_per_cmolc), 'Cs-137 ratio': ratio.reindex(ids).to_numpy()}
{k: int(np.isfinite(v).sum()) for k, v in targets.items()}

{'Kex': 1388, 'Cs-137 ratio': 1389}

The field, soil type and field type of each sample, and the preprocessed spectra:

In [ ]:
#| fukudata
fields = props['field_no'].astype(str).to_numpy()
soil, fieldtype = props['Soil_type_describe'], props['fieldtype']
X_d = preprocessings()['snv_d1'].fit_transform(fuku.spectra)
len(ids), len(np.unique(fields)), X_d.shape

(1389, 391, (1389, 1675))

In [ ]:
#| fukudata
set_seed(0)
models = {'OSSL Kex': load_pretrained('ossl_kex_snv_d1'), 'OSSL CEC': load_pretrained('ossl_cec_snv_d1'), 'random': new_model()}
B4 = {m: layer_features(model, X_d)['block 4'] for m, model in models.items()}
feats = {'spectra': X_d, 'block 4, OSSL Kex': B4['OSSL Kex'], 'block 4, OSSL Kex + CEC': np.hstack([B4['OSSL Kex'], B4['OSSL CEC']]), 'block 4, random': B4['random']}
{k: v.shape for k, v in feats.items()}

{'spectra': (1389, 1675),
 'block 4, OSSL Kex': (1389, 2560),
 'block 4, OSSL Kex + CEC': (1389, 5120),
 'block 4, random': (1389, 2560)}

## Grouped and shuffled cross-validation

PLSR on the spectra, with 1 to 60 components, and ridge regressions on every set of features, with the penalties of `probe` in `07_inside_transfer`. Each runs with folds grouped by field and with shuffled folds, which treat every sample as its own group, inner folds included:

In [ ]:
#| fukudata
ridge = (make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': np.logspace(-1, 6, 15)})
plsr = (PLSRegression(scale=False), {'n_components': range(1, 61)})
runs = {'PLSR, spectra': (plsr, 'spectra'), **{f'ridge, {k}': (ridge, k) for k in feats}}
schemes = {'grouped by field': fields, 'shuffled': np.asarray(ids)}
list(runs), list(schemes)

(['PLSR, spectra',
  'ridge, spectra',
  'ridge, block 4, OSSL Kex',
  'ridge, block 4, OSSL Kex + CEC',
  'ridge, block 4, random'],
 ['grouped by field', 'shuffled'])

Out-of-fold predictions of every run, target and scheme, and their scores over all samples:

In [ ]:
#| fukudata
oof, chosen = {}, {}
for t, y in targets.items():
    ok = np.isfinite(y)
    for s, grp in schemes.items():
        for name, ((est, grid), fk) in runs.items():
            oof[t, s, name] = np.full(len(y), np.nan)
            oof[t, s, name][ok], chosen[t, s, name] = oof_preds(est, grid, feats[fk][ok], y[ok], group_folds(grp[ok]), grp[ok])
pd.DataFrame({k: plain_scores(targets[k[0]][np.isfinite(p)], p[np.isfinite(p)]) for k, p in oof.items()}).T[['R²', 'r', 'RMSE']].unstack(1).round(3)

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. Thi

R²  ...     RMSE
                                            grouped by field  ... shuffled
Cs-137 ratio PLSR, spectra                             0.515  ...    0.227
             ridge, block 4, OSSL Kex                  0.557  ...    0.211
             ridge, block 4, OSSL Kex + CEC            0.571  ...    0.205
             ridge, block 4, random                    0.516  ...    0.215
             ridge, spectra                            0.524  ...    0.227
Kex          PLSR, spectra                             0.265  ...    0.186
             ridge, block 4, OSSL Kex                  0.252  ...    0.183
             ridge, block 4, OSSL Kex + CEC            0.264  ...    0.181
             ridge, block 4, random                    0.241  ...    0.187
             ridge, spectra                            0.279  ...    0.184

[10 rows x 6 columns]

The chosen ridge penalties and PLSR components, by fold:

In [ ]:
#| fukudata
pd.Series({k: [list(c.values())[0] for c in v] for k, v in chosen.items()}).unstack(1)

grouped by field                                                                                  shuffled
Cs-137 ratio PLSR, spectra                                                                                                   [21, 24, 22, 22, 18]                                                                      [24, 24, 27, 30, 28]
             ridge, block 4, OSSL Kex                                                        [3162.2776601683795, 1000.0, 1000.0, 1000.0, 1000.0]              [1000.0, 316.22776601683796, 316.22776601683796, 1000.0, 316.22776601683796]
             ridge, block 4, OSSL Kex + CEC                                                  [3162.2776601683795, 1000.0, 1000.0, 1000.0, 1000.0]                                                  [1000.0, 1000.0, 1000.0, 1000.0, 1000.0]
             ridge, block 4, random                                    [316.22776601683796, 100.0, 100.0, 316.22776601683796, 316.22776601683796]                                                       [100.0, 100.0, 100.0, 100.0, 100.0]
             ridge, spectra                  [316.22776601683796, 316.22776601683796, 316.22776601683796, 316.22776601683796, 316.22776601683796]                             [100.0, 316.22776601683796, 100.0, 100.0, 316.22776601683796]
Kex          PLSR, spectra                                                                                                   [18, 14, 19, 21, 24]                                                                      [25, 18, 24, 28, 26]
             ridge, block 4, OSSL Kex                                [1000.0, 3162.2776601683795, 3162.2776601683795, 1000.0, 3162.2776601683795]                                                  [1000.0, 1000.0, 1000.0, 1000.0, 1000.0]
             ridge, block 4, OSSL Kex + CEC  [3162.2776601683795, 3162.2776601683795, 3162.2776601683795, 3162.2776601683795, 3162.2776601683795]                                      [1000.0, 1000.0, 1000.0, 3162.2776601683795, 1000.0]
             ridge, block 4, random                                                          [1000.0, 1000.0, 1000.0, 316.22776601683796, 1000.0]  [316.22776601683796, 316.22776601683796, 316.22776601683796, 1000.0, 316.22776601683796]
             ridge, spectra                  [316.22776601683796, 316.22776601683796, 316.22776601683796, 316.22776601683796, 316.22776601683796]                                          [100.0, 100.0, 100.0, 316.22776601683796, 100.0]

## By soil type and field type

With folds grouped by field, the R² of each model by soil type, and its gain over ridge on the spectra, with 95% intervals from resampling fields. The 3 samples with no soil type count only in `all`:

In [ ]:
#| fukudata
def by_group(t, labels, scheme='grouped by field'):
    y = targets[t]
    ok = np.isfinite(y) & np.isfinite(oof[t, scheme, 'PLSR, spectra'])
    preds = {n: oof[t, scheme, n][ok] for n in runs}
    return group_scores(y[ok], preds, labels[ok], fields[ok], ref='ridge, spectra')
cols = ['n', 'fields', 'R²', 'R² low', 'R² high', 'RMSE', 'gain', 'gain low', 'gain high']
gk_soil = by_group('Kex', soil)
gk_soil[cols].round(3)

n  ...  gain high
group           model                                 ...           
all             PLSR, spectra                   1388  ...      0.021
                ridge, spectra                  1388  ...      0.000
                ridge, block 4, OSSL Kex        1388  ...      0.022
                ridge, block 4, OSSL Kex + CEC  1388  ...      0.027
                ridge, block 4, random          1388  ...      0.002
Lowland soil    PLSR, spectra                    907  ...     -0.010
                ridge, spectra                   907  ...      0.000
                ridge, block 4, OSSL Kex         907  ...      0.061
                ridge, block 4, OSSL Kex + CEC   907  ...      0.066
                ridge, block 4, random           907  ...      0.027
Andosols        PLSR, spectra                    213  ...      0.098
                ridge, spectra                   213  ...      0.000
                ridge, block 4, OSSL Kex         213  ...      0.066
                ridge, block 4, OSSL Kex + CEC   213  ...      0.054
                ridge, block 4, random           213  ...     -0.023
Forest soil     PLSR, spectra                    137  ...      0.132
                ridge, spectra                   137  ...      0.000
                ridge, block 4, OSSL Kex         137  ...      0.006
                ridge, block 4, OSSL Kex + CEC   137  ...      0.016
                ridge, block 4, random           137  ...      0.090
Peat soil       PLSR, spectra                     66  ...      0.294
                ridge, spectra                    66  ...      0.000
                ridge, block 4, OSSL Kex          66  ...      0.192
                ridge, block 4, OSSL Kex + CEC    66  ...      0.205
                ridge, block 4, random            66  ...      0.081
Red yellow soil PLSR, spectra                     46  ...      0.100
                ridge, spectra                    46  ...      0.000
                ridge, block 4, OSSL Kex          46  ...      0.809
                ridge, block 4, OSSL Kex + CEC    46  ...      0.696
                ridge, block 4, random            46  ...      1.001
Glay soil       PLSR, spectra                     16  ...      0.931
                ridge, spectra                    16  ...      0.000
                ridge, block 4, OSSL Kex          16  ...      0.120
                ridge, block 4, OSSL Kex + CEC    16  ...      0.077
                ridge, block 4, random            16  ...      0.245

[35 rows x 9 columns]

In [ ]:
#| fukudata
gc_soil = by_group('Cs-137 ratio', soil)
gc_soil[cols].round(3)

n  ...  gain high
group           model                                 ...           
all             PLSR, spectra                   1389  ...      0.021
                ridge, spectra                  1389  ...      0.000
                ridge, block 4, OSSL Kex        1389  ...      0.064
                ridge, block 4, OSSL Kex + CEC  1389  ...      0.077
                ridge, block 4, random          1389  ...      0.035
Lowland soil    PLSR, spectra                    907  ...      0.009
                ridge, spectra                   907  ...      0.000
                ridge, block 4, OSSL Kex         907  ...      0.076
                ridge, block 4, OSSL Kex + CEC   907  ...      0.090
                ridge, block 4, random           907  ...      0.058
Andosols        PLSR, spectra                    213  ...      0.040
                ridge, spectra                   213  ...      0.000
                ridge, block 4, OSSL Kex         213  ...      0.326
                ridge, block 4, OSSL Kex + CEC   213  ...      0.374
                ridge, block 4, random           213  ...      0.164
Forest soil     PLSR, spectra                    138  ...      0.221
                ridge, spectra                   138  ...      0.000
                ridge, block 4, OSSL Kex         138  ...      0.104
                ridge, block 4, OSSL Kex + CEC   138  ...      0.127
                ridge, block 4, random           138  ...      0.112
Peat soil       PLSR, spectra                     66  ...      0.349
                ridge, spectra                    66  ...      0.000
                ridge, block 4, OSSL Kex          66  ...      0.228
                ridge, block 4, OSSL Kex + CEC    66  ...      0.218
                ridge, block 4, random            66  ...      0.236
Red yellow soil PLSR, spectra                     46  ...      0.260
                ridge, spectra                    46  ...      0.000
                ridge, block 4, OSSL Kex          46  ...      0.133
                ridge, block 4, OSSL Kex + CEC    46  ...      0.126
                ridge, block 4, random            46  ...      0.041
Glay soil       PLSR, spectra                     16  ...      0.135
                ridge, spectra                    16  ...      0.000
                ridge, block 4, OSSL Kex          16  ...      2.213
                ridge, block 4, OSSL Kex + CEC    16  ...      1.871
                ridge, block 4, random            16  ...      0.352

[35 rows x 9 columns]

The same by field type:

In [ ]:
#| fukudata
gk_ft, gc_ft = by_group('Kex', fieldtype), by_group('Cs-137 ratio', fieldtype)
pd.concat({'Kex': gk_ft, 'Cs-137 ratio': gc_ft})[cols].round(3)

n  ...  gain high
             group        model                                 ...           
Kex          all          PLSR, spectra                   1388  ...      0.021
                          ridge, spectra                  1388  ...      0.000
                          ridge, block 4, OSSL Kex        1388  ...      0.022
                          ridge, block 4, OSSL Kex + CEC  1388  ...      0.027
                          ridge, block 4, random          1388  ...      0.002
             paddy_field  PLSR, spectra                    880  ...      0.011
                          ridge, spectra                   880  ...      0.000
                          ridge, block 4, OSSL Kex         880  ...      0.073
                          ridge, block 4, OSSL Kex + CEC   880  ...      0.086
                          ridge, block 4, random           880  ...      0.035
             upland_field PLSR, spectra                    505  ...      0.082
                          ridge, spectra                   505  ...      0.000
                          ridge, block 4, OSSL Kex         505  ...      0.023
                          ridge, block 4, OSSL Kex + CEC   505  ...      0.023
                          ridge, block 4, random           505  ...      0.004
Cs-137 ratio all          PLSR, spectra                   1389  ...      0.021
                          ridge, spectra                  1389  ...      0.000
                          ridge, block 4, OSSL Kex        1389  ...      0.064
                          ridge, block 4, OSSL Kex + CEC  1389  ...      0.077
                          ridge, block 4, random          1389  ...      0.035
             paddy_field  PLSR, spectra                    881  ...      0.034
                          ridge, spectra                   881  ...      0.000
                          ridge, block 4, OSSL Kex         881  ...      0.064
                          ridge, block 4, OSSL Kex + CEC   881  ...      0.085
                          ridge, block 4, random           881  ...      0.046
             upland_field PLSR, spectra                    505  ...      0.032
                          ridge, spectra                   505  ...      0.000
                          ridge, block 4, OSSL Kex         505  ...      0.101
                          ridge, block 4, OSSL Kex + CEC   505  ...      0.107
                          ridge, block 4, random           505  ...      0.063

[30 rows x 9 columns]

Soil type and field type overlap, so a difference between field types can come from their soil types and the other way round. The two largest soil types by field type, for the two models that matter most here:

In [ ]:
#| fukudata
both = soil.where(soil.isin(['Lowland soil', 'Andosols'])) + ', ' + fieldtype
keep = ['ridge, spectra', 'ridge, block 4, OSSL Kex + CEC']
pd.concat({t: by_group(t, both).loc[lambda d: d.index.get_level_values(1).isin(keep)] for t in targets})[cols].round(3)

n  ...  gain high
             group                      model                                 ...           
Kex          all                        ridge, spectra                  1388  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC  1388  ...      0.027
             Lowland soil, paddy_field  ridge, spectra                   618  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   618  ...      0.146
             Lowland soil, upland_field ridge, spectra                   286  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   286  ...      0.046
             Andosols, paddy_field      ridge, spectra                   124  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   124  ...      0.123
             Andosols, upland_field     ridge, spectra                    89  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC    89  ...      0.131
Cs-137 ratio all                        ridge, spectra                  1389  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC  1389  ...      0.077
             Lowland soil, paddy_field  ridge, spectra                   618  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   618  ...      0.102
             Lowland soil, upland_field ridge, spectra                   286  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   286  ...      0.134
             Andosols, paddy_field      ridge, spectra                   124  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC   124  ...      0.254
             Andosols, upland_field     ridge, spectra                    89  ...      0.000
                                        ridge, block 4, OSSL Kex + CEC    89  ...      0.836

[20 rows x 9 columns]

For comparison, the same table by soil type with shuffled folds. The difference from the grouped table shows where knowing the field helped most:

In [ ]:
#| fukudata
pd.concat({t: by_group(t, soil, 'shuffled').loc[lambda d: d.index.get_level_values(1).isin(keep)] for t in targets})[cols].round(3)

n  ...  gain high
             group           model                                 ...           
Kex          all             ridge, spectra                  1388  ...      0.000
                             ridge, block 4, OSSL Kex + CEC  1388  ...      0.055
             Lowland soil    ridge, spectra                   907  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   907  ...      0.062
             Andosols        ridge, spectra                   213  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   213  ...      0.157
             Forest soil     ridge, spectra                   137  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   137  ...      0.090
             Peat soil       ridge, spectra                    66  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    66  ...      0.285
             Red yellow soil ridge, spectra                    46  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    46  ...      0.556
             Glay soil       ridge, spectra                    16  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    16  ...      0.741
Cs-137 ratio all             ridge, spectra                  1389  ...      0.000
                             ridge, block 4, OSSL Kex + CEC  1389  ...      0.107
             Lowland soil    ridge, spectra                   907  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   907  ...      0.119
             Andosols        ridge, spectra                   213  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   213  ...      0.351
             Forest soil     ridge, spectra                   138  ...      0.000
                             ridge, block 4, OSSL Kex + CEC   138  ...      0.202
             Peat soil       ridge, spectra                    66  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    66  ...      0.242
             Red yellow soil ridge, spectra                    46  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    46  ...      0.122
             Glay soil       ridge, spectra                    16  ...      0.000
                             ridge, block 4, OSSL Kex + CEC    16  ...      1.456

[28 rows x 9 columns]

## Findings

Knowing the field inflated every earlier score. With shuffled folds, which like the Cs137 study's split put most fields in both training and test, ridge on the spectra reaches R² 0.38 for Kex and 0.58 for the Cs-137 ratio. With folds grouped by field, it reaches 0.28 and 0.52. Ridge on block 4 of the OSSL Kex and CEC models drops more, from 0.40 to 0.26 for Kex and from 0.66 to 0.57 for the ratio.

For Kex, the OSSL features add nothing on new fields. Over all samples, their R² gain over ridge on the spectra is -0.015 (95% interval -0.055 to 0.027). With shuffled folds it is 0.019 (-0.017 to 0.055). So the small OSSL gain in the earlier notebooks partly came from recognising fields.

For the Cs-137 ratio, frozen OSSL features help on new fields, but only slightly. All three models below use ridge regression, scored with folds grouped by field; only the input changes. Ridge on the preprocessed spectra (SNV, then a Savitzky-Golay first derivative) gives R² 0.52. Ridge on block 4 of the OSSL Kex and CEC models, with their weights frozen, gives R² 0.57, a gain of 0.047 (0.017 to 0.077). Ridge on block 4 of the same models with random weights gains -0.009. The gain therefore comes from OSSL pretraining, not from the network architecture. It cuts the unexplained variance from 48% to 43%. With shuffled folds, the random-weights model gains too. This revises `10_cs137`, where the random model took most of the gain on a split that shares fields.

No soil type holds the weak Kex transfer on its own. OSSL features give no clear gain in any soil type, and their gain is above zero only in Lowland paddy fields: 0.079 (0.015 to 0.146), from an R² of 0.02 to 0.10. In Andosols, which OSSL's Andisols might be expected to help, the gain is -0.056 (-0.199 to 0.054).

Kex is barely predictable within paddy fields. On new fields, ridge on the spectra gives R² -0.03 in paddy fields (880 samples) and 0.20 in upland fields (505). In Lowland paddy fields it gives 0.02, and in Andosol paddy fields -0.39. Much of the overall R² of 0.28 therefore comes from telling paddy from upland samples, not from ranking samples within either. Paddy fields hold 63% of the samples. Whether their Kex reflects added K fertiliser, which leaves no mineral signature in the spectra, is a question for the Fukushima data team.

The Cs-137 ratio is predictable in every large group, with R² 0.48 to 0.60 on new fields in paddy and upland fields and in Lowland soils. The OSSL gain is about 0.05 in each of these groups. It is largest in Andosols, at 0.12 (-0.035 to 0.374) on new fields and 0.17 (0.069 to 0.351) with shuffled folds. PLSR beats ridge on the spectra in Forest soils, by 0.11 (0.047 to 0.221).

Groups of fewer than about 70 samples, which are Peat, Red-yellow and Gley soils, have R² intervals too wide to compare models. Their percentile intervals can even leave out the point estimate, because R² on a resample of a few fields is biased low.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()